In [ ]:
# import requests
# import json
# import base64
# from pathlib import Path

# API_KEY_REF = 'your-openrouter-api-key-here'
# def encode_pdf_to_base64(pdf_path):
#     with open(pdf_path, "rb") as pdf_file:
#         return base64.b64encode(pdf_file.read()).decode('utf-8')

# url = "https://openrouter.ai/api/v1/chat/completions"
# headers = {
#     "Authorization": f"Bearer {API_KEY_REF}",
#     "Content-Type": "application/json"
# }

# # Read and encode the PDF
# pdf_path = "data/~Moran~K~19910201~Spirometry Exam~20250729~20250729032843.pdf"
# base64_pdf = encode_pdf_to_base64(pdf_path)
# data_url = f"data:application/pdf;base64,{base64_pdf}"

# messages = [
#     {
#         "role": "user",
#         "content": [
#             {
#                 "type": "text",
#                 "text": "Please extract the Spirometry table from the pdf and return the values in csv format, "
#                 "note that it is the unit of parameter that is beside it and it should not be a column. "
#                 "The '-' Should be treated as empty values."
#                 "do not add 'csv' at the start or end of the response"
#             },
#             {
#                 "type": "file",
#                 "file": {
#                     "filename": "document.pdf",
#                     "file_data": data_url
#                 }
#             },
#         ]
#     }
# ]

# # Optional: Configure PDF processing engine
# # PDF parsing will still work even if the plugin is not explicitly set
# plugins = [
#     {
#         "id": "file-parser",
#         "pdf": {
#             "engine": "pdf-text"  # defaults to "mistral-ocr". See Pricing above
#         }
#     }
# ]

# payload = {
#     "model": "google/gemini-2.5-flash-lite",
#     "messages": messages,
# }

# response = requests.post(url, headers=headers, json=payload)
# # Get the response content
# response_data = response.json()
# print(response_data)

# # Extract the content from the response
# if 'choices' in response_data and len(response_data['choices']) > 0:
#     content = response_data['choices'][0]['message']['content']
    
#     # Save to a CSV file
#     output_file = "extracted_table.csv"
#     with open(output_file, 'w', encoding='utf-8') as f:
#         f.write(content)
    
#     print(f"Content saved to {output_file}")
# else:
#     print("No content found in response")

In [2]:
import pandas as pd
import os

base_dir = os.path.dirname(os.path.abspath('.'))
spirometry_df = pd.read_csv(f"{base_dir}/data/extracted_spirometry_table.csv")

fvc_best = spirometry_df.loc[spirometry_df['Parameters'] == 'FVC', 'Best'].values[0]
fvc_pred = spirometry_df.loc[spirometry_df['Parameters'] == 'FVC', '%Pred.'].values[0]

fev1_best = spirometry_df.loc[spirometry_df['Parameters'] == 'FEV1', 'Best'].values[0]
fev1_pred = spirometry_df.loc[spirometry_df['Parameters'] == 'FEV1', '%Pred.'].values[0]

fev1_fevc_best = spirometry_df.loc[spirometry_df['Parameters'] == 'FEV1/FVC%', 'Best'].values[0]
fev1_fevc_pred = spirometry_df.loc[spirometry_df['Parameters'] == 'FEV1/FVC%', '%Pred.'].values[0]

print(f"FVC Best: {fvc_best}, FVC Pred: {fvc_pred}")
print(f"FEV1 Best: {fev1_best}, FEV1 Pred: {fev1_pred}")
print(f"FEV1/FVC% Best: {fev1_fevc_best}, FEV1/FVC% Pred: {fev1_fevc_pred}")

FVC Best: 2.95, FVC Pred: 90.6
FEV1 Best: 2.35, FEV1 Pred: 90.3
FEV1/FVC% Best: 79.39, FEV1/FVC% Pred: 99.1


In [25]:
df = pd.read_csv(f'{base_dir}/data/Pnoe_20251124_1109-Jadidi_Parastou 4.csv', delimiter=';')
peak_vt = df['VT(l)'].max()
max_vt_row = df.loc[df['VT(l)'].idxmax()]
print(f"Peak VT: {peak_vt}")
bf = max_vt_row['BF(bpm)']
print(f"BF at Peak VT: {bf}")
hr = max_vt_row['HR(bpm)']
print(f"HR at Peak VT: {hr}")

answer = peak_vt*60/bf
print(f"Answer (Peak VT * 60 / BF): {answer}")

Peak VT: 2.67
BF at Peak VT: 30.21
HR at Peak VT: 126.0
Answer (Peak VT * 60 / BF): 5.3028798411122136


In [31]:
df = pd.read_csv(f'{base_dir}/data/Pnoe_20251124_1109-Jadidi_Parastou 4.csv', delimiter=';')
# Convert all columns to numeric where possible, coercing errors to NaN
df = df.apply(pd.to_numeric, errors='ignore')
df['VO2 Pulse'] = df['VO2(ml/min)'] / df['HR(bpm)']  # VO2 Pulse in mL/beat
df['VO2 Breath'] = df['VO2(ml/min)'] / df['BF(bpm)']  # VO2 per Breath in mL/breath
df['CHO'] = df['EE(kcal/min)'] * df['CARBS(%)']/100
df['FAT'] = df['EE(kcal/min)'] * df['FAT(%)']/100
# Smooth key columns using rolling window
window_size = 5

# List of columns to smooth
columns_to_smooth = ['VO2(ml/min)', 'VCO2(ml/min)', 'HR(bpm)', 'VT(l)', 'BF(bpm)', 'VE(l/min)', 'VO2 Pulse', 'VO2 Breath', 'CHO', 'FAT']

# Apply smoothing to each column
for col in columns_to_smooth:
    if col in df.columns:
        df[f'{col}_smoothed'] = df[col].rolling(window=window_size).mean()
        
peak_vt = df['VT(l)_smoothed'].max()
max_vt_row = df.loc[df['VT(l)_smoothed'].idxmax()]
print(f"Peak VT: {peak_vt}")
hr = max_vt_row['HR(bpm)_smoothed']
bf = max_vt_row['BF(bpm)_smoothed']
print(f"BF at Peak VT: {bf}")
print(f"HR at Peak VT: {hr}")

answer = peak_vt*60/bf
print(f"Answer (Peak VT * 60 / BF): {answer}")

Peak VT: 2.148
BF at Peak VT: 32.016000000000005
HR at Peak VT: 127.05
Answer (Peak VT * 60 / BF): 4.025487256371814


/tmp/ipykernel_190586/2895637222.py:3: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')


In [22]:
df.columns

Index(['T(sec)', 'PHASE', 'HR(bpm)', 'VO2(ml/min)', 'VCO2(ml/min)', 'RER',
       'VE(l/min)', 'FEO2', 'FECO2', 'FETO2', 'FETCO2', 'PETO2 (mmHg)',
       'PETCO2(mmHg)', 'FIO2', 'FICO2', 'VT(l)', 'BF(bpm)', 'EE(kcal/day)',
       'EE(kcal/min)', 'CARBS(kcal)', 'CARBS(%)', 'FAT(kcal)', 'FAT(%)', 'MET',
       'CUMULATIVE EE(kcal)', 'BP(kPa)', 'Watts', 'Speed', 'VO2 Pulse',
       'VO2 Breath', 'CHO', 'FAT', 'VO2(ml/min)_smoothed',
       'VCO2(ml/min)_smoothed', 'HR(bpm)_smoothed', 'VT(l)_smoothed',
       'BF(bpm)_smoothed', 'VE(l/min)_smoothed', 'VO2 Pulse_smoothed',
       'VO2 Breath_smoothed', 'CHO_smoothed', 'FAT_smoothed', 'fat_carb_ratio',
       'vo2_breath_slope', 'vo2_breath_slope_smoothed', 'vo2_pulse_slope',
       'hr_slope', 'slope_difference', 'slope_difference_smoothed'],
      dtype='object')

In [5]:
percent_fev = (peak_vt / fev1_best) * 100
print(f"Percent FEV: {percent_fev}")

Percent FEV: 86.68085106382978


In [6]:
personal_df = pd.read_excel(f'{base_dir}/data/SECA body comp for all patients.xlsx')

keirstyn_data = personal_df[personal_df['LastName'].str.contains('Moran', case=False, na=False)]
keirstyn_data

,MeasurementDate,Comment,ExternalDeviceId,ExternalPatientId,FirstName,LastName,BirthDate,Age,Ethnicity,Gender,...,Child_XC,Child_XC_Unit,Child_BIVA_ZRh,Child_BIVA_ZXcH,Child_PhA,Child_PhA_Unit,Child_REE_Kcal,Child_REE_MJ,Child_TEE_Kcal,Child_TEE_MJ
13,2025-07-29T18:58:54.0000000Z,NaN,10000001583275_0055003f5631501320313557,KM6479696509,Keirstyn,Moran,1991-02-01T00:00:00.0000000Z,34,Caucasian,Female,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
v02_max = df['VO2(ml/min)_smoothed'].max()
weight = keirstyn_data['Weight'].iloc[0]
print(f"VO2 Max: {v02_max/weight}")

VO2 Max: 57.06306451612903


In [8]:
# Find the point where fat burning is highest and carb burning is lowest
# Using the smoothed data for more stable results
fat_burn_max_idx = df['FAT_smoothed'].idxmax()
carb_burn_min_idx = df['CHO_smoothed'].idxmin()

# # Get the data at maximum fat burning point
# max_fat_row = df.loc[fat_burn_max_idx]
# print(f"Maximum Fat Burning Point:")
# print(f"Time: {max_fat_row['T(sec)']} seconds")
# print(f"Fat burn rate: {max_fat_row['FAT_smoothed']:.3f} kcal/min")
# print(f"Carb burn rate: {max_fat_row['CHO_smoothed']:.3f} kcal/min")
# print(f"Heart Rate: {max_fat_row['HR(bpm)_smoothed']:.1f} bpm")
# print(f"VO2: {max_fat_row['VO2(ml/min)_smoothed']:.1f} ml/min")

# print("\n" + "="*50)

# # Get the data at minimum carb burning point
# min_carb_row = df.loc[carb_burn_min_idx]
# print(f"Minimum Carbohydrate Burning Point:")
# print(f"Time: {min_carb_row['T(sec)']} seconds")
# print(f"Fat burn rate: {min_carb_row['FAT_smoothed']:.3f} kcal/min")
# print(f"Carb burn rate: {min_carb_row['CHO_smoothed']:.3f} kcal/min")
# print(f"Heart Rate: {min_carb_row['HR(bpm)_smoothed']:.1f} bpm")
# print(f"VO2: {min_carb_row['VO2(ml/min)_smoothed']:.1f} ml/min")

print("\n" + "="*50)

# Find the optimal fat burning zone (highest fat:carb ratio)
df['fat_carb_ratio'] = df['FAT_smoothed'] / (df['CHO_smoothed'] + 0.00000001)  # Add small value to avoid division by zero
optimal_fat_idx = df['fat_carb_ratio'].idxmax()
optimal_row = df.loc[optimal_fat_idx]

print("Optimal Fat Burning Zone (highest fat:carb ratio):")
print(f"Time: {optimal_row['T(sec)']} seconds")
print(f"Fat burn rate: {optimal_row['FAT_smoothed']:.3f} kcal/min")
print(f"Carb burn rate: {optimal_row['CHO_smoothed']:.3f} kcal/min")
print(f"Fat:Carb ratio: {optimal_row['fat_carb_ratio']:.2f}")
print(f"Heart Rate: {optimal_row['HR(bpm)_smoothed']:.1f} bpm")
print(f"VO2: {optimal_row['VO2(ml/min)_smoothed']:.1f} ml/min")


Optimal Fat Burning Zone (highest fat:carb ratio):
Time: 155.0 seconds
Fat burn rate: 4.429 kcal/min
Carb burn rate: 1.343 kcal/min
Fat:Carb ratio: 3.30
Heart Rate: 94.2 bpm
VO2: 1217.4 ml/min


In [9]:
# Find intersections where FAT_smoothed and CHO_smoothed cross each other
intersections = []

for i in range(1, len(df)):
    # Check if there's a crossover between consecutive points
    prev_fat = df.iloc[i-1]['FAT_smoothed']
    prev_cho = df.iloc[i-1]['CHO_smoothed']
    curr_fat = df.iloc[i]['FAT_smoothed']
    curr_cho = df.iloc[i]['CHO_smoothed']
    
    # Skip if any values are NaN
    if pd.isna(prev_fat) or pd.isna(prev_cho) or pd.isna(curr_fat) or pd.isna(curr_cho):
        continue
    
    # Check if lines cross (fat was above/below cho and now it's below/above)
    if ((prev_fat > prev_cho and curr_fat < curr_cho) or 
        (prev_fat < prev_cho and curr_fat > curr_cho)):
        intersections.append(i)

print(f"Found {len(intersections)} intersections at indices: {intersections}")

if intersections:
    # Get the last intersection
    last_intersection_idx = intersections[-1]
    last_intersection_row = df.iloc[last_intersection_idx]
    
    print(f"\nLast intersection at index {last_intersection_idx}:")
    print(f"Time: {last_intersection_row['T(sec)']} seconds")
    print(f"Fat burn rate: {last_intersection_row['FAT_smoothed']:.3f} kcal/min")
    print(f"Carb burn rate: {last_intersection_row['CHO_smoothed']:.3f} kcal/min")
    print(f"Heart Rate: {last_intersection_row['HR(bpm)_smoothed']:.1f} bpm")
    print(f"VO2: {last_intersection_row['VO2(ml/min)_smoothed']:.1f} ml/min")
else:
    print("No intersections found between FAT and CHO curves")

Found 6 intersections at indices: [18, 19, 20, 44, 59, 60]

Last intersection at index 60:
Time: 297.0 seconds
Fat burn rate: 3.065 kcal/min
Carb burn rate: 3.259 kcal/min
Heart Rate: 95.0 bpm
VO2: 1306.7 ml/min


In [10]:
def detect_vt1(df, fat_col="FAT_smoothed", carb_col="CHO_smoothed"):
    """
    Detect VT1 as the first index where carb burn > fat burn and remains higher.
    """
    condition = df[carb_col] > df[fat_col]
    crossover_indices = condition[condition].index

    if len(crossover_indices) == 0:
        return None  # No crossover found
    
    # Find first crossover where carbs remain higher for the rest
    for idx in crossover_indices:
        if all(df.loc[idx:][carb_col] > df.loc[idx:][fat_col]):
            return idx
    return None


def detect_vt2(df, vent_col="VE(l/min)_smoothed", bf_col="BF(bpm)_smoothed", smooth_window=5):
    """
    Detect VT2 using slope/inflection method.
    Works with either Ventilation (VE) or Breathing Frequency (Bf).
    """
    col = vent_col if vent_col in df.columns else bf_col
    
    # Use already smoothed data
    smoothed_col = col
    
    # Compute slope (first derivative)
    df["slope"] = df[smoothed_col].diff()
    
    # Detect inflection: largest change in slope (second derivative peak)
    df["second_derivative"] = df["slope"].diff()
    inflection_idx = df["second_derivative"].idxmax()
    
    return inflection_idx


def analyze_thresholds(df_input):
    # Use the existing dataframe
    df_copy = df_input.copy()
    
    # --- Detect VT1 ---
    vt1_idx = detect_vt1(df_copy)
    vt1 = None
    if vt1_idx is not None:
        vt1 = {
            "HeartRate": df_copy.loc[vt1_idx, "HR(bpm)_smoothed"],
            "Speed": df_copy.loc[vt1_idx, "Speed"],
            "Time": df_copy.loc[vt1_idx, "T(sec)"]
        }
    
    # --- Detect VT2 ---
    vt2_idx = detect_vt2(df_copy)
    vt2 = None
    if vt2_idx is not None:
        vt2 = {
            "HeartRate": df_copy.loc[vt2_idx, "HR(bpm)_smoothed"],
            "Speed": df_copy.loc[vt2_idx, "Speed"],
            "Time": df_copy.loc[vt2_idx, "T(sec)"]
        }
    
    return vt1, vt2


vt1, vt2 = analyze_thresholds(df)
print("VT1:", vt1)
print("VT2:", vt2)


VT1: {'HeartRate': 95.0, 'Speed': 4.0, 'Time': 297.0}
VT2: {'HeartRate': 156.9, 'Speed': 6.0, 'Time': 1477.0}


In [11]:
zone_1_start = optimal_row['HR(bpm)_smoothed'] - 15
zone_2_start = optimal_row['HR(bpm)_smoothed']
zone_3_start = vt1
zone_4_start = vt2['HeartRate'] - 10
zone_5_start = vt2['HeartRate']
zone_5_end = vt2['HeartRate'] + 10

zone_1_end = zone_2_start
zone_2_end = vt1['HeartRate']
zone_3_end = zone_4_start
zone_4_end = zone_5_start

print(f"Zone 1 (Active Recovery): {zone_1_start:.1f} - {zone_1_end:.1f} bpm")
print(f"Zone 2 (Aerobic Base): {zone_2_start:.1f} - {zone_2_end:.1f} bpm")
print(f"Zone 3 (Aerobic): {zone_3_start['HeartRate']:.1f} - {zone_3_end:.1f} bpm")
print(f"Zone 4 (Lactate Threshold): {zone_4_start:.1f} - {zone_4_end:.1f} bpm")
print(f"Zone 5 (VO2 Max): {zone_5_start:.1f} - {zone_5_end:.1f} bpm")

Zone 1 (Active Recovery): 79.2 - 94.2 bpm
Zone 2 (Aerobic Base): 94.2 - 95.0 bpm
Zone 3 (Aerobic): 95.0 - 146.9 bpm
Zone 4 (Lactate Threshold): 146.9 - 156.9 bpm
Zone 5 (VO2 Max): 156.9 - 166.9 bpm


In [12]:
# Calculate the slope of VO2 Breath (first derivative)
df['vo2_breath_slope'] = df['VO2 Breath_smoothed'].diff()

# Find points where slope is consistently zero or negative
# We'll use a rolling window to check for consistent negative/zero slope
window = len(df) // 3  # Number of consecutive points to check

# Calculate rolling mean of slope to smooth out noise
df['vo2_breath_slope_smoothed'] = df['vo2_breath_slope'].rolling(window=window).mean()

# Find where slope becomes consistently zero or negative
mask = df['vo2_breath_slope_smoothed'] <= 0
consistent_negative_indices = mask[mask].index

if len(consistent_negative_indices) > 0:
    # Find the first point where slope becomes consistently negative/zero
    vo2_max_idx = consistent_negative_indices[0]
    vo2_max_row = df.loc[vo2_max_idx]
    
    print(f"VO2 Max detected at index {vo2_max_idx}:")
    print(f"Time: {vo2_max_row['T(sec)']} seconds")
    print(f"VO2 Breath: {vo2_max_row['VO2 Breath_smoothed']:.1f} ml/breath")
    print(f"VO2: {vo2_max_row['VO2(ml/min)_smoothed']:.1f} ml/min")
    print(f"VO2 per kg: {vo2_max_row['VO2(ml/min)_smoothed']/weight:.1f} ml/kg/min")
    print(f"Heart Rate: {vo2_max_row['HR(bpm)_smoothed']:.1f} bpm")
    print(f"Speed: {vo2_max_row['Speed']} km/h")
    print(f"VO2 Breath Slope: {vo2_max_row['vo2_breath_slope_smoothed']:.2f}")
else:
    # If no consistent negative slope found, use the maximum VO2 Breath value
    vo2_max_idx = df['VO2 Breath_smoothed'].idxmax()
    vo2_max_row = df.loc[vo2_max_idx]
    
    print(f"No consistent negative slope found. Using peak VO2 Breath at index {vo2_max_idx}:")
    print(f"Time: {vo2_max_row['T(sec)']} seconds")
    print(f"VO2 Breath: {vo2_max_row['VO2 Breath_smoothed']:.1f} ml/breath")
    print(f"VO2: {vo2_max_row['VO2(ml/min)_smoothed']:.1f} ml/min")
    print(f"VO2 per kg: {vo2_max_row['VO2(ml/min)_smoothed']/weight:.1f} ml/kg/min")
    print(f"Heart Rate: {vo2_max_row['HR(bpm)_smoothed']:.1f} bpm")
    print(f"Speed: {vo2_max_row['Speed']} km/h")

VO2 Max detected at index 395:
Time: 1780.0 seconds
VO2 Breath: 52.1 ml/breath
VO2: 2728.1 ml/min
VO2 per kg: 48.9 ml/kg/min
Heart Rate: 168.9 bpm
Speed: 0.0 km/h
VO2 Breath Slope: -0.03


In [13]:
# Calculate slopes for both VO2 Pulse and HR
df['vo2_pulse_slope'] = df['VO2 Pulse_smoothed'].diff()
df['hr_slope'] = df['HR(bpm)_smoothed'].diff()

# Calculate the difference between the slopes
df['slope_difference'] = abs(df['vo2_pulse_slope'] - df['hr_slope'])

# Find where the slope difference becomes consistently large (slopes diverge)
# Use a rolling window to smooth out noise
window_size = 10  # Adjust window size as needed
df['slope_difference_smoothed'] = df['slope_difference'].rolling(window=window_size).mean()

# Find the threshold - we'll use the 75th percentile of slope differences as threshold
threshold = df['slope_difference_smoothed'].quantile(0.75)

# Find points where slope difference exceeds threshold
divergence_mask = df['slope_difference_smoothed'] > threshold
divergence_indices = divergence_mask[divergence_mask].index

if len(divergence_indices) > 0:
    # Find the first sustained divergence point
    min_consecutive_points = 5
    consistent_divergence_idx = None
    
    for start_idx in divergence_indices:
        # Check if divergence is sustained for consecutive points
        consecutive_count = 0
        for j in range(start_idx, min(start_idx + min_consecutive_points, len(df))):
            if j in divergence_indices:
                consecutive_count += 1
            else:
                break
        
        if consecutive_count >= min_consecutive_points:
            consistent_divergence_idx = start_idx
            break
    
    if consistent_divergence_idx is not None:
        divergence_row = df.iloc[consistent_divergence_idx]
        
        print(f"VO2 Pulse and HR slopes diverge consistently starting at index {consistent_divergence_idx}:")
        print(f"Time: {divergence_row['T(sec)']} seconds")
        print(f"VO2 Pulse (smoothed): {divergence_row['VO2 Pulse_smoothed']:.2f}")
        print(f"Heart Rate (smoothed): {divergence_row['HR(bpm)_smoothed']:.1f} bpm")
        print(f"VO2 Pulse Slope: {divergence_row['vo2_pulse_slope']:.3f}")
        print(f"HR Slope: {divergence_row['hr_slope']:.3f}")
        print(f"Slope Difference: {divergence_row['slope_difference_smoothed']:.3f}")
        print(f"VO2: {divergence_row['VO2(ml/min)_smoothed']:.1f} ml/min")
        print(f"Speed: {divergence_row['Speed']} km/h")
        print(f"Threshold used: {threshold:.3f}")
    else:
        print(f"No sustained divergence found. Threshold: {threshold:.3f}")
        # Show the point with maximum slope difference instead
        max_diff_idx = df['slope_difference_smoothed'].idxmax()
        max_diff_row = df.iloc[max_diff_idx]
        
        print(f"\nPoint with maximum slope difference at index {max_diff_idx}:")
        print(f"Time: {max_diff_row['T(sec)']} seconds")
        print(f"VO2 Pulse (smoothed): {max_diff_row['VO2 Pulse_smoothed']:.2f}")
        print(f"Heart Rate (smoothed): {max_diff_row['HR(bpm)_smoothed']:.1f} bpm")
        print(f"Slope Difference: {max_diff_row['slope_difference_smoothed']:.3f}")
else:
    print("No significant slope divergence found between VO2 Pulse and HR")

VO2 Pulse and HR slopes diverge consistently starting at index 19:
Time: 93.0 seconds
VO2 Pulse (smoothed): 7.72
Heart Rate (smoothed): 85.1 bpm
VO2 Pulse Slope: -0.389
HR Slope: 2.600
Slope Difference: 1.577
VO2: 641.7 ml/min
Speed: 3.5 km/h
Threshold used: 0.429


In [14]:
df['HR(bpm)_smoothed'].max()

168.9

In [15]:
96.7/197

0.4908629441624366

In [16]:
max_fat_smoothed_idx = df['FAT_smoothed'].idxmax()
max_fat_smoothed_row = df.loc[max_fat_smoothed_idx]
max_heart_rate = 220 - keirstyn_data['Age'].iloc[0]

print(f"Maximum FAT_smoothed occurs at index {max_fat_smoothed_idx}:")
print(f"Heart Rate (smoothed): {max_fat_smoothed_row['HR(bpm)_smoothed']:.1f} bpm")
print(f"FAT (smoothed): {max_fat_smoothed_row['FAT_smoothed']:.3f} kcal/min")

Maximum FAT_smoothed occurs at index 29:
Heart Rate (smoothed): 94.2 bpm
FAT (smoothed): 4.429 kcal/min


In [17]:
# Step 1: Filter resting phase (usually lowest VO2 or MET values)
rest_phase = df[df['MET'] <= 1.1]  # assuming <1.1 MET means rest

# Step 2: Compute resting metabolic rate
rmr = rest_phase['EE(kcal/day)'].mean()

print(f"Estimated RMR from data: {rmr:.0f} kcal/day")


Estimated RMR from data: 1369 kcal/day


In [18]:
rest_phase = df[df['RER'] == 0.9]  # filter rest data
fat_rest = rest_phase['FAT(%)'].mean()
carb_rest = rest_phase['CARBS(%)'].mean()

print(f"Resting phase fuel mix: Fats {fat_rest:.1f}%, Carbs {carb_rest:.1f}%")


Resting phase fuel mix: Fats 33.2%, Carbs 66.8%


In [19]:
import numpy as np
# Convert necessary columns to numeric (errors='coerce' turns non-numeric to NaN)
df['T(sec)'] = pd.to_numeric(df['T(sec)'], errors='coerce')
df['HR(bpm)'] = pd.to_numeric(df['HR(bpm)'], errors='coerce')
df['PHASE'] = df['PHASE'].astype(str) # Ensure PHASE is a string

# Drop any rows where HR or Time is missing after conversion
df.dropna(subset=['T(sec)', 'HR(bpm)'], inplace=True)


# 2. Identify the Peak Heart Rate (HR_Peak)
# This is the maximum HR recorded during the entire test.
HR_peak = df['HR(bpm)_smoothed'].max()


# 3. Find the End-of-Exercise Time (T_End)
# The end of the ramp test is usually the last point where the phase is NOT 'Active Recovery'
# We'll assume the 'Active Recovery' phase starts immediately after the end of the max effort.
# The last entry *before* or *at the start* of the 'Active Recovery' phase is a good proxy for T_End.
# A more conservative and robust method is to find the time corresponding to the HR_peak, but since
# the HR peak often occurs in the last few seconds of the max effort, we'll try to locate the phase change.

# Find the row where 'Active Recovery' begins (first time the PHASE column contains 'Recovery')
recovery_start_row = df[df['PHASE'].str.contains('Recovery', na=False)].iloc[0]
T_recovery_start = recovery_start_row['T(sec)']

# The end of exercise is assumed to be right before the recovery phase starts.
# For simplicity and standard procedure, we will use the time corresponding to the peak HR as the end time.
T_peak = df[df['HR(bpm)_smoothed'] == HR_peak]['T(sec)'].max()


# 4. Find the Heart Rate at 1-minute Post-Exercise (HR_60s)
# This is the heart rate at T_peak + 60 seconds.

# The target time is 60 seconds after the peak HR time.
T_target_60s = T_peak + 60

# Find the row with the time closest to T_target_60s
# We use 'np.argmin' to find the index of the minimum absolute difference
idx_60s = np.argmin(np.abs(df['T(sec)'] - T_target_60s))
HR_60s = df.iloc[idx_60s]['HR(bpm)']
T_found_60s = df.iloc[idx_60s]['T(sec)']


# 5. Calculate Heart Rate Recovery (HRR)
HRR_60s = HR_peak - HR_60s

print(f"--- Heart Rate Recovery (HRR @ 1 Minute) ---")
print(f"Peak Heart Rate (HR_Peak):      {HR_peak:.2f} bpm (at T={T_peak:.2f}s)")
print(f"HR 1-Minute Post-Exercise (HR_60s): {HR_60s:.2f} bpm (at T={T_found_60s:.2f}s)")
print(f"Calculated HRR:                  {HRR_60s:.2f} bpm")

print(HRR_60s/HR_peak)

--- Heart Rate Recovery (HRR @ 1 Minute) ---
Peak Heart Rate (HR_Peak):      168.90 bpm (at T=1780.00s)
HR 1-Minute Post-Exercise (HR_60s): 120.00 bpm (at T=1841.00s)
Calculated HRR:                  48.90 bpm
0.28952042628774427


In [20]:
import numpy as np
# Convert necessary columns to numeric (errors='coerce' turns non-numeric to NaN)
df['T(sec)'] = pd.to_numeric(df['T(sec)'], errors='coerce')
df['HR(bpm)'] = pd.to_numeric(df['HR(bpm)'], errors='coerce')
df['PHASE'] = df['PHASE'].astype(str) # Ensure PHASE is a string

# Drop any rows where HR or Time is missing after conversion
df.dropna(subset=['T(sec)', 'HR(bpm)'], inplace=True)


# 2. Identify the Peak Heart Rate (HR_Peak)
# This is the maximum HR recorded during the entire test.
HR_peak = df['HR(bpm)_smoothed'].max()


# 3. Find the End-of-Exercise Time (T_End)
# The end of the ramp test is usually the last point where the phase is NOT 'Active Recovery'
# We'll assume the 'Active Recovery' phase starts immediately after the end of the max effort.
# The last entry *before* or *at the start* of the 'Active Recovery' phase is a good proxy for T_End.
# A more conservative and robust method is to find the time corresponding to the HR_peak, but since
# the HR peak often occurs in the last few seconds of the max effort, we'll try to locate the phase change.

# Find the row where 'Active Recovery' begins (first time the PHASE column contains 'Recovery')
recovery_start_row = df[df['PHASE'].str.contains('Recovery', na=False)].iloc[0]
T_recovery_start = recovery_start_row['T(sec)']

# The end of exercise is assumed to be right before the recovery phase starts.
# For simplicity and standard procedure, we will use the time corresponding to the peak HR as the end time.
T_peak = df[df['HR(bpm)_smoothed'] == HR_peak]['T(sec)'].max()


# 4. Find the Heart Rate at 1-minute Post-Exercise (HR_60s)
# This is the heart rate at T_peak + 60 seconds.

# The target time is 60 seconds after the peak HR time.
T_target_150s = T_peak + 150

# Find the row with the time closest to T_target_150s
# We use 'np.argmin' to find the index of the minimum absolute difference
idx_150s = np.argmin(np.abs(df['T(sec)'] - T_target_150s))
bf_150s = df.iloc[idx_150s]['BF(bpm)']
T_found_150s = df.iloc[idx_150s]['T(sec)']

first_breath = df['BF(bpm)'].min()
print(f"BF_150: {bf_150s:.2f} bpm (at T={T_found_150s:.2f}s)")
print(f"First breath: {first_breath:.2f} bpm")
print(f"Percentage: {(bf_150s - first_breath) / first_breath * 100:.2f}%")

BF_150: 10.01 bpm (at T=1910.00s)
First breath: 8.40 bpm
Percentage: 19.17%


In [21]:
df.columns

Index(['T(sec)', 'PHASE', 'HR(bpm)', 'VO2(ml/min)', 'VCO2(ml/min)', 'RER',
       'VE(l/min)', 'FEO2', 'FECO2', 'FETO2', 'FETCO2', 'PETO2 (mmHg)',
       'PETCO2(mmHg)', 'FIO2', 'FICO2', 'VT(l)', 'BF(bpm)', 'EE(kcal/day)',
       'EE(kcal/min)', 'CARBS(kcal)', 'CARBS(%)', 'FAT(kcal)', 'FAT(%)', 'MET',
       'CUMULATIVE EE(kcal)', 'BP(kPa)', 'Watts', 'Speed', 'VO2 Pulse',
       'VO2 Breath', 'CHO', 'FAT', 'VO2(ml/min)_smoothed',
       'VCO2(ml/min)_smoothed', 'HR(bpm)_smoothed', 'VT(l)_smoothed',
       'BF(bpm)_smoothed', 'VE(l/min)_smoothed', 'VO2 Pulse_smoothed',
       'VO2 Breath_smoothed', 'CHO_smoothed', 'FAT_smoothed', 'fat_carb_ratio',
       'vo2_breath_slope', 'vo2_breath_slope_smoothed', 'vo2_pulse_slope',
       'hr_slope', 'slope_difference', 'slope_difference_smoothed'],
      dtype='object')